In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Apache Spark uses Java, so first we must install that
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-8-jdk-headless -qq

# Unpack Spark from google drive
!tar xzf /content/drive/MyDrive/spark-3.3.0-bin-hadoop3.tgz

# Set up environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "spark-3.3.0-bin-hadoop3"

# Install findspark, which helps python locate the psyspark module files
!pip install -q findspark
import findspark
findspark.init()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# Restarting the Spark session to ensure environment variables are correctly mapped to the JVM
try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession
spark = SparkSession.builder\
        .master("local")\
        .appName("Colab")\
        .config('spark.ui.port', '4050')\
        .getOrCreate()

from pyspark.sql import functions as F
from pyspark.sql.functions import col

In [ ]:
# 1. Download the Heterogeneity Activity Recognition dataset
!wget -q https://archive.ics.uci.edu/static/public/344/heterogeneity+activity+recognition.zip -O har_hetero.zip

# 2. Unzip the main download
!unzip -q -o har_hetero.zip -d har_hetero_extracted

# 3. Unzip the nested zip files found inside the extracted folder
!unzip -q -o "/content/har_hetero_extracted/Activity recognition exp.zip" -d /content/har_hetero_extracted/data
!unzip -q -o "/content/har_hetero_extracted/Still exp.zip" -d /content/har_hetero_extracted/data

# 4. Load the dataset using PySpark with recursive lookup
har_hetero_df = spark.read\
    .option('header', 'True')\
    .option('inferSchema', 'True')\
    .option('recursiveFileLookup', 'True')\
    .csv('/content/har_hetero_extracted/data/')

# Show results
har_hetero_df.printSchema()
har_hetero_df.show(20)

root
 |-- Index: string (nullable = true)
 |-- Arrival_Time: string (nullable = true)
 |-- Creation_Time: string (nullable = true)
 |-- x: string (nullable = true)
 |-- y: string (nullable = true)
 |-- z: string (nullable = true)
 |-- User: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Device: string (nullable = true)
 |-- gt: string (nullable = true)

+-----+-------------+-------------------+--------------------+--------------------+--------------------+----+------+--------+-----+
|Index| Arrival_Time|      Creation_Time|                   x|                   y|                   z|User| Model|  Device|   gt|
+-----+-------------+-------------------+--------------------+--------------------+--------------------+----+------+--------+-----+
|    0|1424696633909|1424696631914042029|         0.013748169|-0.00062561035000...|        -0.023376465|   a|nexus4|nexus4_1|stand|
|    1|1424696633909|1424696631919046912|0.014816283999999999|       -0.0016937256|         -0.0

In [ ]:
har_hetero_df.printSchema()

root
 |-- Index: string (nullable = true)
 |-- Arrival_Time: string (nullable = true)
 |-- Creation_Time: string (nullable = true)
 |-- x: string (nullable = true)
 |-- y: string (nullable = true)
 |-- z: string (nullable = true)
 |-- User: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Device: string (nullable = true)
 |-- gt: string (nullable = true)



In [ ]:
!ls -lh har_hetero.zip /content/har_hetero_extracted/

-rw-r--r-- 1 root root 785M Jun 13 14:21 har_hetero.zip

/content/har_hetero_extracted/:
total 785M
-rwx------ 1 root root 742M May 22  2023 'Activity recognition exp.zip'
drwxr-xr-x 5 root root 4.0K Jun 13 14:22  data
-rwx------ 1 root root  43M May 22  2023 'Still exp.zip'


# **TASK 2**

In [ ]:
# Exploring data to determine ingestion and preprocessing strategy
print(f"Current Partition Count: {har_hetero_df.rdd.getNumPartitions()}")
print(f"Total Row Count: {har_hetero_df.count()}")
har_hetero_df.select('x', 'y', 'z', 'gt').show(5)

Current Partition Count: 33
Total Row Count: 36349875
+--------------------+--------------------+------------+-----+
|                   x|                   y|           z|   gt|
+--------------------+--------------------+------------+-----+
|         0.013748169|-0.00062561035000...|-0.023376465|stand|
|0.014816283999999999|       -0.0016937256| -0.02230835|stand|
|           0.0158844|       -0.0016937256|-0.021240234|stand|
|         0.016952515|        -0.003829956| -0.02017212|stand|
|           0.0158844|-0.00703430180000...| -0.02017212|stand|
+--------------------+--------------------+------------+-----+
only showing top 5 rows



In [ ]:
# Code Cell 7: Data Preprocessing
from pyspark.sql.functions import col

# 1. Cast numeric columns to DoubleType as they were inferred as strings
# 2. Handle missing values by dropping rows where the target 'gt' is null
# Explanation: Numerical casting is required for scaling; removing nulls ensures model stability.
df_preprocessed = har_hetero_df.withColumn("x", col("x").cast("double")) \
    .withColumn("y", col("y").cast("double")) \
    .withColumn("z", col("z").cast("double")) \
    .dropna(subset=["gt"])

df_preprocessed.printSchema()

root
 |-- Index: string (nullable = true)
 |-- Arrival_Time: string (nullable = true)
 |-- Creation_Time: string (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- z: double (nullable = true)
 |-- User: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Device: string (nullable = true)
 |-- gt: string (nullable = true)



In [ ]:
# Code Cell 8: Feature Engineering Steps
# To resolve the 'Encountered null' error, we should ensure the input data is clean.

from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.sql.functions import col

# 1. Clean the data: Drop rows with nulls in feature columns ('x', 'y', 'z') and target column ('gt')
# Note: We include 'gt' because StringIndexer will also fail if it encounters null labels.
# (Assuming your DataFrame is named `df`, update if necessary)
df_preprocessed = df_preprocessed.dropna(subset=['x', 'y', 'z', 'gt'])

# 2. Verification Check: Assert that no nulls remain in ['x', 'y', 'z']
null_count = df_preprocessed.filter(col("x").isNull() | col("y").isNull() | col("z").isNull()).count()
assert null_count == 0, f"Preprocessing Check Failed: Found {null_count} rows with nulls in x, y, or z."
print("Data is clean. No null values found in x, y, or z.")

# Encoding: StringIndexer converts categorical labels (gt) into numerical indices
indexer = StringIndexer(inputCol="gt", outputCol="label")

# Transformation: VectorAssembler combines features into a single vector column
# This is now guaranteed to succeed without the 'Encountered null' error.
assembler = VectorAssembler(inputCols=["x", "y", "z"], outputCol="features")

# Scaling: StandardScaler normalizes features to have unit standard deviation
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)

Data is clean. No null values found in x, y, or z.


In [ ]:
# Code Cell 9: PySpark Pipeline Construction
from pyspark.ml import Pipeline

# Constructing the pipeline stage by order:
# 1. StringIndexer (Labeling)
# 2. VectorAssembler (Feature grouping)
# 3. StandardScaler (Normalization)
pipeline = Pipeline(stages=[indexer, assembler, scaler])

# Note: This failed because df_preprocessed still contains nulls in x, y, or z.
# Before running this again, please update the dropna() in the Preprocessing cell
# to include your feature columns, then re-run that cell and this one.
try:
    pipeline_model = pipeline.fit(df_preprocessed)
    data_final = pipeline_model.transform(df_preprocessed)
    data_final.select("scaledFeatures", "label").show(5)
except Exception as e:
    print("Still encountering an error. Check if your preprocessing step removed all nulls from ['x', 'y', 'z'].")

+--------------------+-----+
|      scaledFeatures|label|
+--------------------+-----+
|[0.00373789558391...|  3.0|
|[0.00402829806162...|  3.0|
|[0.00431870081122...|  3.0|
|[0.00460910328893...|  3.0|
|[0.00431870081122...|  3.0|
+--------------------+-----+
only showing top 5 rows



In [ ]:
# Code Cell 10: Data Ingestion with Partition Count
# Re-partitioning to 100 to better distribute the 36M rows across the cluster
df_ingested = har_hetero_df.repartition(100)

print(f"New Partition Count: {df_ingested.rdd.getNumPartitions()}")

New Partition Count: 100


In [ ]:
# DATA PREPROCESSING & SPLITTING
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

# 1. First, split the clean preprocessed data into raw train and test sets
# We use df_preprocessed from the cell where you cleaned the nulls
train_df_raw, test_df_raw = df_preprocessed.randomSplit([0.8, 0.2], seed=42)

# 2. Fit the StringIndexer to the training data to create the model
# 'indexer' was defined in the previous Feature Engineering cell
indexer_model = indexer.fit(train_df_raw)

# 3. Apply transformations (Indexing and Assembling)
train_df = assembler.transform(indexer_model.transform(train_df_raw))
test_df = assembler.transform(indexer_model.transform(test_df_raw))

# 4. Fit and apply the Scaler
# We fit on train and transform both to prevent data leakage
scaler_model = scaler.fit(train_df)
train_df = scaler_model.transform(train_df)
test_df = scaler_model.transform(test_df)

print("Data preprocessing complete.")
print(f"Training set size: {train_df.count()}")
print(f"Test set size: {test_df.count()}")

Data preprocessing complete.
Training set size: 26992837
Test set size: 6748663


In [ ]:
# EVALUATION & METRICS HELPER
import time
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder # Changed from CrossValidator
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier

def train_and_evaluate(model_name, estimator, param_grid, train_data, test_data):
    evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

    # OPTIMIZATION 1: Use TrainValidationSplit instead of CrossValidator
    # This evaluates each parameter combination ONCE (e.g., 80/20 split) instead of 3 times.
    # This alone will cut your hyperparameter tuning time by ~66%.
    tvs = TrainValidationSplit(estimator=estimator,
                               estimatorParamMaps=param_grid,
                               evaluator=evaluator_f1,
                               trainRatio=0.8,  # 80% for training, 20% for validation
                               parallelism=4)   # Evaluate 4 parameter combinations in parallel

    start_time = time.time()
    tvs_model = tvs.fit(train_data)
    training_time = time.time() - start_time

    best_model = tvs_model.bestModel
    predictions = best_model.transform(test_data)

    # Calculate Standard Metrics
    accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(predictions)
    precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision").evaluate(predictions)
    recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall").evaluate(predictions)
    f1 = evaluator_f1.evaluate(predictions)

    # OPTIMIZATION 2: Sample the predictions before converting to Pandas for AUC-ROC
    # Converting millions of rows with probability vectors to Pandas will crash your driver or take forever.
    preds_sample = predictions.select("label", "probability").sample(0.1, seed=42).toPandas()
    y_true = preds_sample['label']
    y_prob = np.stack(preds_sample['probability'].apply(lambda x: x.toArray()))

    num_classes = int(train_data.select("label").agg({"label": "max"}).collect()[0][0]) + 1
    try:
        auc_roc = roc_auc_score(y_true, y_prob, multi_class='ovr', labels=list(range(num_classes)))
    except Exception as e:
        print(f"AUC-ROC calculation skipped due to: {e}")
        auc_roc = None

    # Extract Best Hyperparameters
    best_param_map = tvs_model.getEstimatorParamMaps()[np.argmax(tvs_model.validationMetrics)]
    best_params = {param.name: val for param, val in best_param_map.items()}

    return {
        "Algorithm": model_name,
        "Hyperparameters": str(best_params),
        "Training Time (s)": round(training_time, 2),
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4),
        "AUC-ROC": round(auc_roc, 4) if auc_roc else "N/A",
        "Model_Object": best_model
    }

def print_model_summary(model, name):
    print(f"\n--- {name} Summary ---")
    if name == "Logistic Regression":
        try:
            print(f"Objective History (first 5): {model.summary.objectiveHistory[:5]}")
        except: print("Summary unavailable for TVS wrapper.")
    elif name in ["Random Forest", "GBT", "Decision Tree"]:
        print(f"Feature Importances: {model.featureImportances}")
    print("-" * 30)

In [ ]:
# IMPLEMENTATION OF FOUR ML MODELS
from pyspark.ml.classification import MultilayerPerceptronClassifier
import polars as pl
import IPython

# Updated summary function to handle the new MLP model
def print_model_summary(model, name):
    print(f"\n--- {name} Summary ---")
    if name == "Logistic Regression":
        try:
            print(f"Objective History (first 5): {model.summary.objectiveHistory[:5]}")
        except: print("Summary unavailable for TVS wrapper.")
    elif name in ["Random Forest", "Decision Tree"]:
        print(f"Feature Importances: {model.featureImportances}")
    elif name == "Multilayer Perceptron":
        print(f"Layer Architecture: {model.getLayers()}")
    print("-" * 30)

print("Caching training and test data...")
train_df.cache()
test_df.cache()

# Trigger an action to materialize the cache
train_rows = train_df.count()
test_rows = test_df.count()
print(f"Data cached. Original Train rows: {train_rows}, Test rows: {test_rows}")

# 27 Million rows will cause OutOfMemory errors or infinite training times for complex models.
# We sample 10% of the data to ensure the notebook finishes running successfully.
train_df = train_df.sample(0.1, seed=42)
test_df = test_df.sample(0.1, seed=42)
print(f"Sampled data. New Train rows: {train_df.count()}, New Test rows: {test_df.count()}")

# Determine number of classes for MLP
num_classes = int(train_df.select("label").agg({"label": "max"}).collect()[0][0]) + 1
# Architecture: [Input features (x,y,z), Hidden Layer 1, Hidden Layer 2, Output classes]
layers = [3, 8, 6, num_classes]

model_configs = [
    ("Logistic Regression",
     LogisticRegression(featuresCol='scaledFeatures', labelCol='label', maxIter=30),
     ParamGridBuilder().addGrid(LogisticRegression.regParam, [0.1]).build()),

    ("Decision Tree",
     DecisionTreeClassifier(featuresCol='scaledFeatures', labelCol='label', maxDepth=5),
     ParamGridBuilder().addGrid(DecisionTreeClassifier.maxDepth, [5, 10]).build()),

    ("Random Forest",
     RandomForestClassifier(featuresCol='scaledFeatures', labelCol='label', numTrees=20),
     ParamGridBuilder().addGrid(RandomForestClassifier.numTrees, [20]).build()),

    ("Multilayer Perceptron",
     MultilayerPerceptronClassifier(featuresCol='scaledFeatures', labelCol='label', layers=layers, blockSize=128, seed=42, maxIter=20),
     ParamGridBuilder().addGrid(MultilayerPerceptronClassifier.stepSize, [0.05, 0.1]).build())
]

results_list = []

for name, estimator, grid in model_configs:
    print(f"Starting training and tuning for: {name}")
    result = train_and_evaluate(name, estimator, grid, train_df, test_df)
    results_list.append(result)
    print_model_summary(result['Model_Object'], name)

# Display the final summary table using Polars
summary_df = pl.DataFrame(results_list).drop('Model_Object')
display(summary_df)

Caching training and test data...
Data cached. Original Train rows: 26992837, Test rows: 6748663
Sampled data. New Train rows: 2701899, New Test rows: 675304
Starting training and tuning for: Logistic Regression

--- Logistic Regression Summary ---
Objective History (first 5): [1.942500031045654, 1.9145540877096456, 1.9042432439389056, 1.9041399999116038, 1.9041191175688683]
------------------------------
Starting training and tuning for: Decision Tree

--- Decision Tree Summary ---
Feature Importances: (3,[0,1,2],[0.42162454289009726,0.17867902017311252,0.39969643693679036])
------------------------------
Starting training and tuning for: Random Forest

--- Random Forest Summary ---
Feature Importances: (3,[0,1,2],[0.4502208216479179,0.1140307936067964,0.43574838474528577])
------------------------------
Starting training and tuning for: Multilayer Perceptron

--- Multilayer Perceptron Summary ---
Layer Architecture: [3, 8, 6, 7]
------------------------------


Algorithm,Hyperparameters,Training Time (s),Accuracy,Precision,Recall,F1-Score,AUC-ROC
str,str,f64,f64,f64,f64,f64,f64
"""Logistic Regression""","""{'regParam': 0.1}""",193.33,0.2179,0.1528,0.2179,0.1476,0.586
"""Decision Tree""","""{'maxDepth': 5}""",251.7,0.3663,0.344,0.3663,0.3032,0.7503
"""Random Forest""","""{'numTrees': 20}""",328.26,0.3717,0.3553,0.3717,0.3116,0.7639
"""Multilayer Perceptron""","""{'stepSize': 0.05}""",553.15,0.2525,0.1427,0.2525,0.1702,0.5894
